# Notebook 48 — Meta-Law for Coefficient Dynamics

This notebook learns a predictive law for the **coefficient vector itself**, mapping regime metadata
to the coefficients of the sparse governing equation family.

We keep the fixed template from Notebook 47:

\[
g(r,c;\beta) = \beta_0 + \beta_1 c + \beta_2 r + \beta_3 c^3 + \beta_4 r^2 + \beta_5 r c^2
\]

and now model:

\[
\beta = h(\text{system}, \text{task}, \text{forcing mode}, k, \text{flow mode})
\]

## Main goals

1. Predict coefficient vectors from metadata alone.
2. Test whether latent-space prediction is easier than raw coefficient prediction.
3. Reconstruct governing fields and trajectories from predicted coefficients.
4. Compare **shared-law**, **meta-law**, and **regime-specific** fits.

In [ ]:
# Core imports

import warnings
warnings.filterwarnings("ignore")

import os
import glob
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import r2_score, mean_squared_error
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neural_network import MLPRegressor

try:
    from IPython.display import display
except Exception:
    display = print

np.random.seed(42)

In [ ]:
# Data discovery and synthetic fallback

DATA_PATH = None

def autodetect_data_path():
    candidates = []
    patterns = [
        "*residual*flow*.parquet",
        "*residual*flow*.csv",
        "*governing*flow*.parquet",
        "*governing*flow*.csv",
        "*.parquet",
        "*.csv",
        "*.pkl",
        "*.pickle",
    ]
    for pat in patterns:
        candidates.extend(glob.glob(pat))
        candidates.extend(glob.glob(os.path.join("data", pat)))
        candidates.extend(glob.glob(os.path.join("outputs", pat)))
    candidates = [c for c in candidates if os.path.isfile(c)]
    return candidates[0] if candidates else None

def synthetic_dataset():
    systems = ["entropy", "unevenness"]
    tasks = ["zeta_vs_gue", "zeta_vs_poisson"]
    forcing_modes = ["capacity_gap", "feature_gap", "condition_gap"]
    ks = [3, 5, 7]
    flow_modes = ["linear", "nonlinear"]

    rows = []
    sample_id = 0
    for system in systems:
        for task in tasks:
            for forcing_mode in forcing_modes:
                for k in ks:
                    for flow_mode in flow_modes:
                        n_paths = 14
                        n_steps = 42
                        c_grid = np.linspace(-1.25, 1.05, n_steps)

                        sys_shift = 0.06 if system == "entropy" else -0.04
                        task_shift = 0.05 if task == "zeta_vs_gue" else -0.03
                        force_shift = {"capacity_gap": 0.00, "feature_gap": 0.03, "condition_gap": 0.08}[forcing_mode]
                        k_shift = {3: -0.04, 5: 0.02, 7: 0.05}[k]
                        nl_gain = 1.0 if flow_mode == "nonlinear" else 0.55

                        for path_id in range(n_paths):
                            r = -0.75 + 0.10 * path_id + 0.05 * np.sin(0.7 * path_id)
                            for window_id, c in enumerate(c_grid):
                                g = (
                                    0.58 * np.tanh(1.35 * c)
                                    + 0.42 * c
                                    - 0.78 * r
                                    + 0.20 * r**2
                                    + nl_gain * 0.07 * c**2
                                    + nl_gain * 0.10 * r * c
                                    - nl_gain * 0.025 * r**3
                                    + sys_shift + task_shift + force_shift + k_shift
                                )
                                if forcing_mode == "condition_gap":
                                    g += 0.06 * np.sin(2.3 * c)
                                if system == "entropy":
                                    g += 0.03 * np.cos(1.2 * c)
                                if flow_mode == "linear":
                                    g -= 0.09 * r**2
                                    g += 0.015 * c * r

                                delta_c = c_grid[min(window_id + 1, n_steps - 1)] - c if window_id < n_steps - 1 else c - c_grid[max(window_id - 1, 0)]
                                noise = 0.012 * np.random.randn()
                                next_r = r + (g + noise) * delta_c

                                rows.append(
                                    {
                                        "system": system,
                                        "task": task,
                                        "forcing_mode": forcing_mode,
                                        "k": k,
                                        "flow_mode": flow_mode,
                                        "condition_coord": c,
                                        "residual": r,
                                        "predicted_flow": g + noise,
                                        "next_residual": next_r,
                                        "delta_condition": delta_c,
                                        "sample_id": sample_id,
                                        "path_id": path_id,
                                        "window_id": window_id,
                                        "jacobian_norm": abs(-0.78 + 0.40 * r + nl_gain * 0.10 * c - 0.075 * r**2),
                                        "integration_error": abs(noise),
                                    }
                                )
                                r = next_r
                                sample_id += 1
    return pd.DataFrame(rows)

if DATA_PATH is None:
    DATA_PATH = autodetect_data_path()

USE_SYNTHETIC_FALLBACK = DATA_PATH is None
print("Selected DATA_PATH:", DATA_PATH)
print("USE_SYNTHETIC_FALLBACK:", USE_SYNTHETIC_FALLBACK)

In [ ]:
# Loading + schema alignment

def load_dataframe(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".parquet":
        return pd.read_parquet(path)
    if ext in [".pkl", ".pickle"]:
        return pd.read_pickle(path)
    return pd.read_csv(path)

alias_groups = {
    "condition_coord": ["condition_coord", "c", "condition", "cond", "condition_coordinate"],
    "residual": ["residual", "r", "resid"],
    "predicted_flow": ["predicted_flow", "flow", "g", "drdc", "delta_residual_per_condition"],
    "next_residual": ["next_residual", "r_next", "next_r"],
    "delta_condition": ["delta_condition", "dc", "delta_c"],
    "forcing_mode": ["forcing_mode", "forcing", "forcing_gap", "mode"],
}

def align_schema(df):
    cols = list(df.columns)
    rename_map = {}
    for canonical, aliases in alias_groups.items():
        for a in aliases:
            if a in cols and a != canonical:
                rename_map[a] = canonical
                break
    df = df.rename(columns=rename_map)

    if "next_residual" not in df.columns and {"residual", "predicted_flow", "delta_condition"}.issubset(df.columns):
        df["next_residual"] = df["residual"] + df["predicted_flow"] * df["delta_condition"]

    if "delta_condition" not in df.columns and "condition_coord" in df.columns:
        df = df.sort_values(["condition_coord"]).copy()
        dc = np.gradient(df["condition_coord"].to_numpy())
        df["delta_condition"] = dc

    defaults = {
        "system": "unknown_system",
        "task": "unknown_task",
        "forcing_mode": "unknown_forcing",
        "k": 5,
        "flow_mode": "unknown_flow",
        "sample_id": np.arange(len(df)),
        "path_id": 0,
        "window_id": np.arange(len(df)),
    }
    for k, v in defaults.items():
        if k not in df.columns:
            df[k] = v

    required = ["condition_coord", "residual", "predicted_flow"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns after alignment: {missing}")

    return df

if USE_SYNTHETIC_FALLBACK:
    df = synthetic_dataset()
    data_source = "synthetic_fallback"
else:
    df = align_schema(load_dataframe(DATA_PATH))
    data_source = DATA_PATH

df = align_schema(df)
print("Data source:", data_source)
print("Shape:", df.shape)
display(df.head())

## Build per-regime coefficient dataset

Each regime is defined by:

- `system`
- `task`
- `forcing_mode`
- `k`
- `flow_mode`

For each regime we fit the fixed template law and store:
- the coefficient vector
- fit quality
- additive fraction
- forward/backward asymmetry

In [ ]:
# Fixed template functions

TERM_NAMES = ["1", "c", "r", "c^3", "r^2", "r c^2"]
COEF_COLS = [f"coef_{t}" for t in TERM_NAMES]

def design_template(data):
    r = data["residual"].to_numpy(dtype=float)
    c = data["condition_coord"].to_numpy(dtype=float)
    return np.column_stack([
        np.ones_like(r),
        c,
        r,
        c**3,
        r**2,
        r * c**2,
    ])

def fit_template(sub, alpha=1e-6):
    X = design_template(sub)
    y = sub["predicted_flow"].to_numpy(dtype=float)
    beta = np.linalg.solve(X.T @ X + alpha * np.eye(X.shape[1]), X.T @ y)
    pred = X @ beta

    out = {"n": len(sub), "r2": r2_score(y, pred), "rmse": math.sqrt(mean_squared_error(y, pred))}
    for name, coef in zip(TERM_NAMES, beta):
        out[f"coef_{name}"] = float(coef)
    return beta, pred, out

def predict_with_beta(sub, beta):
    return design_template(sub) @ beta

def additive_fraction(sub):
    g = sub[["condition_coord", "residual", "predicted_flow"]].dropna().copy()
    g["c_bin"] = pd.cut(g["condition_coord"], bins=20, labels=False, include_lowest=True)
    g["r_bin"] = pd.cut(g["residual"], bins=20, labels=False, include_lowest=True)
    grouped = g.groupby(["c_bin", "r_bin"]).agg(mean_flow=("predicted_flow", "mean")).reset_index()
    grid = grouped.pivot(index="r_bin", columns="c_bin", values="mean_flow").sort_index()
    G = grid.values
    overall = np.nanmean(G)
    col_eff = np.nanmean(G - overall, axis=0)
    row_eff = np.nanmean(G - overall, axis=1)
    G_add = overall + row_eff[:, None] + col_eff[None, :]
    denom = np.nansum((G - np.nanmean(G))**2)
    if denom <= 0:
        return np.nan
    return 1 - np.nansum((G - G_add)**2) / denom

def fb_gap_metric(sub, beta, n_r0=18):
    cmin, cmax = sub["condition_coord"].min(), sub["condition_coord"].max()
    rmin, rmax = sub["residual"].min(), sub["residual"].max()
    cgrid = np.linspace(cmin, cmax, 45)
    r0s = np.linspace(np.quantile(sub["residual"], 0.05), np.quantile(sub["residual"], 0.95), n_r0)
    flow_cap = max(1.0, 2.5 * np.quantile(np.abs(sub["predicted_flow"]), 0.995))

    def g_of(r, c):
        x = np.array([1.0, c, r, c**3, r**2, r * c**2], dtype=float)
        val = float(x @ beta)
        return float(np.clip(val, -flow_cap, flow_cap))

    def integrate(r0, direction="forward"):
        use_c = cgrid if direction == "forward" else cgrid[::-1]
        r = float(np.clip(r0, rmin, rmax))
        traj = [r]
        for i in range(len(use_c) - 1):
            c = use_c[i]
            dc = float(use_c[i + 1] - use_c[i])
            r = float(r + g_of(r, c) * dc)
            r = float(np.clip(r, rmin - 0.5, rmax + 0.5))
            traj.append(r)
        traj = np.array(traj)
        if direction == "backward":
            traj = traj[::-1]
        return traj

    f_terms, b_terms = [], []
    for r0 in r0s:
        tf = integrate(r0, "forward")
        tb = integrate(r0, "backward")
        f_terms.append(tf[-1])
        b_terms.append(tb[0])
    return float(np.mean(np.abs(np.array(f_terms) - np.array(b_terms))))

In [ ]:
# Per-regime coefficient dataset

rows = []
regime_subsets = {}

group_cols = ["system", "task", "forcing_mode", "k", "flow_mode"]
for vals, sub in df.groupby(group_cols):
    if len(sub) < 30:
        continue
    beta, pred, fit_stats = fit_template(sub.copy())
    kval = int(vals[3]) if float(vals[3]).is_integer() else vals[3]
    regime_id = f"{vals[0]}|{vals[1]}|{vals[2]}|k={kval}|{vals[4]}"
    row = {
        "regime_id": regime_id,
        "system": vals[0],
        "task": vals[1],
        "forcing_mode": vals[2],
        "k": float(vals[3]),
        "flow_mode": vals[4],
        "additive_r2": additive_fraction(sub),
        "fb_gap": fb_gap_metric(sub, beta),
    }
    row.update(fit_stats)
    rows.append(row)
    regime_subsets[regime_id] = sub.copy()

coef_df = pd.DataFrame(rows).reset_index(drop=True)
display(coef_df.head())
print("Number of regimes:", len(coef_df))

## Metadata encoding

We encode regime descriptors and then predict:
- either the raw coefficient vector
- or low-dimensional PCA coordinates of the coefficient manifold

In [ ]:
# Metadata design matrix

meta_cols_cat = ["system", "task", "forcing_mode", "flow_mode"]

meta_df = coef_df[["regime_id"] + meta_cols_cat + ["k"]].copy()
meta_X = pd.get_dummies(meta_df[meta_cols_cat], drop_first=False)
meta_X["k"] = coef_df["k"].astype(float).values
meta_X["k2"] = coef_df["k"].astype(float).values ** 2

X_meta = meta_X.copy()
Y_coef = coef_df[COEF_COLS].copy()

print("Metadata design shape:", X_meta.shape)
display(X_meta.head())
display(Y_coef.head())

## Direct coefficient prediction

We compare simple meta-models:
- linear
- ridge
- small MLP

The notebook uses leave-one-regime-out prediction to stay honest at small sample sizes.

In [ ]:
# Leave-one-regime-out predictions for coefficient vectors

def fit_predict_leave_one_out(X_df, Y_df, model_builder):
    X = X_df.to_numpy(dtype=float)
    Y = Y_df.to_numpy(dtype=float)
    preds = np.zeros_like(Y, dtype=float)

    for i in range(len(X)):
        mask = np.ones(len(X), dtype=bool)
        mask[i] = False
        model = model_builder()
        model.fit(X[mask], Y[mask])
        preds[i] = model.predict(X[i:i+1])[0]
    return preds

def build_linear():
    return LinearRegression()

def build_ridge():
    return Ridge(alpha=1.0)

class MultiTargetMLP:
    def __init__(self, hidden_layer_sizes=(16,), random_state=42, max_iter=4000):
        self.hidden_layer_sizes = hidden_layer_sizes
        self.random_state = random_state
        self.max_iter = max_iter
        self.models = None

    def fit(self, X, Y):
        Y = np.asarray(Y, dtype=float)
        if Y.ndim == 1:
            Y = Y[:, None]

        self.models = [
            MLPRegressor(
                hidden_layer_sizes=self.hidden_layer_sizes,
                random_state=self.random_state,
                max_iter=self.max_iter
            )
            for _ in range(Y.shape[1])
        ]

        for j, m in enumerate(self.models):
            m.fit(X, Y[:, j])
        return self

    def predict(self, X):
        cols = [m.predict(X) for m in self.models]
        return np.column_stack(cols)

def build_mlp():
    return MultiTargetMLP()

pred_linear = fit_predict_leave_one_out(X_meta, Y_coef, build_linear)
pred_ridge  = fit_predict_leave_one_out(X_meta, Y_coef, build_ridge)
pred_mlp    = fit_predict_leave_one_out(X_meta, Y_coef, build_mlp)

def score_coef_predictions(Y_true, Y_pred, label):
    rows = []
    for j, c in enumerate(COEF_COLS):
        rows.append({
            "model": label,
            "term": c,
            "r2": r2_score(Y_true.iloc[:, j], Y_pred[:, j]),
            "rmse": math.sqrt(mean_squared_error(Y_true.iloc[:, j], Y_pred[:, j])),
        })
    return pd.DataFrame(rows)

score_linear = score_coef_predictions(Y_coef, pred_linear, "linear")
score_ridge  = score_coef_predictions(Y_coef, pred_ridge, "ridge")
score_mlp    = score_coef_predictions(Y_coef, pred_mlp, "mlp_small")
coef_pred_scores = pd.concat([score_linear, score_ridge, score_mlp], ignore_index=True)
display(coef_pred_scores)

In [ ]:
# Plot predicted vs actual coefficients for best direct model by mean RMSE

mean_rmse_by_model = coef_pred_scores.groupby("model")["rmse"].mean().sort_values()
best_direct_model = mean_rmse_by_model.index[0]
pred_map = {"linear": pred_linear, "ridge": pred_ridge, "mlp_small": pred_mlp}
best_pred = pred_map[best_direct_model]

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
axes = axes.ravel()
for ax, j, c in zip(axes, range(len(COEF_COLS)), COEF_COLS):
    ax.scatter(Y_coef.iloc[:, j], best_pred[:, j])
    mn = min(Y_coef.iloc[:, j].min(), best_pred[:, j].min())
    mx = max(Y_coef.iloc[:, j].max(), best_pred[:, j].max())
    ax.plot([mn, mx], [mn, mx], linestyle="--")
    ax.set_title(c)
    ax.set_xlabel("actual")
    ax.set_ylabel("predicted")
fig.suptitle(f"Coefficient prediction — best direct model: {best_direct_model}", y=1.02)
plt.tight_layout()
plt.show()

## Latent-space meta-law

Since Notebook 47 suggested a low-dimensional coefficient manifold, we also:
1. compress coefficients with PCA
2. predict latent coordinates from metadata
3. reconstruct the coefficient vector

In [ ]:
# PCA latent modeling

coef_scaler = StandardScaler()
Y_std = coef_scaler.fit_transform(Y_coef.to_numpy())
pca = PCA(n_components=2, random_state=42)
Z = pca.fit_transform(Y_std)

print("Explained variance ratio:", pca.explained_variance_ratio_)

Z_df = pd.DataFrame(Z, columns=["pc1", "pc2"])

predZ_linear = fit_predict_leave_one_out(X_meta, Z_df, build_linear)
predZ_ridge  = fit_predict_leave_one_out(X_meta, Z_df, build_ridge)
predZ_mlp    = fit_predict_leave_one_out(X_meta, Z_df, build_mlp)

def reconstruct_from_latent(predZ):
    Zhat = np.array(predZ, dtype=float)
    Yhat_std = pca.inverse_transform(Zhat)
    Yhat = coef_scaler.inverse_transform(Yhat_std)
    return Yhat

recon_linear = reconstruct_from_latent(predZ_linear)
recon_ridge  = reconstruct_from_latent(predZ_ridge)
recon_mlp    = reconstruct_from_latent(predZ_mlp)

latent_scores = pd.concat([
    score_coef_predictions(Y_coef, recon_linear, "latent_linear"),
    score_coef_predictions(Y_coef, recon_ridge, "latent_ridge"),
    score_coef_predictions(Y_coef, recon_mlp, "latent_mlp_small"),
], ignore_index=True)
display(latent_scores)

In [ ]:
# Plot true vs predicted latent coordinates for best latent model

latent_mean_rmse = latent_scores.groupby("model")["rmse"].mean().sort_values()
best_latent_model = latent_mean_rmse.index[0]
latent_pred_map = {
    "latent_linear": predZ_linear,
    "latent_ridge": predZ_ridge,
    "latent_mlp_small": predZ_mlp,
}
best_predZ = latent_pred_map[best_latent_model]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for i, pc in enumerate(["pc1", "pc2"]):
    axes[i].scatter(Z_df.iloc[:, i], best_predZ[:, i])
    mn = min(Z_df.iloc[:, i].min(), best_predZ[:, i].min())
    mx = max(Z_df.iloc[:, i].max(), best_predZ[:, i].max())
    axes[i].plot([mn, mx], [mn, mx], linestyle="--")
    axes[i].set_title(pc)
    axes[i].set_xlabel("actual")
    axes[i].set_ylabel("predicted")
fig.suptitle(f"Latent prediction — best latent model: {best_latent_model}", y=1.03)
plt.tight_layout()
plt.show()

## Metadata importance by ablation

We remove one metadata source at a time and re-run a simple ridge meta-model.
This shows which inputs matter for coefficient transport.

In [ ]:
# Ablation tests using ridge on raw coefficients

def run_ablation(drop_columns):
    Xab = X_meta.drop(columns=drop_columns, errors="ignore")
    pred = fit_predict_leave_one_out(Xab, Y_coef, build_ridge)
    rmse = math.sqrt(mean_squared_error(Y_coef.to_numpy().ravel(), pred.ravel()))
    return rmse

ablation_rows = []
baseline_rmse = run_ablation([])

groups_to_drop = {
    "drop_system": [c for c in X_meta.columns if c.startswith("system_")],
    "drop_task": [c for c in X_meta.columns if c.startswith("task_")],
    "drop_forcing_mode": [c for c in X_meta.columns if c.startswith("forcing_mode_")],
    "drop_flow_mode": [c for c in X_meta.columns if c.startswith("flow_mode_")],
    "drop_k": ["k", "k2"],
}

for name, cols in groups_to_drop.items():
    rmse = run_ablation(cols)
    ablation_rows.append({"ablation": name, "rmse": rmse, "delta_vs_full": rmse - baseline_rmse})

ablation_df = pd.DataFrame(ablation_rows).sort_values("delta_vs_full", ascending=False)
display(ablation_df)

In [ ]:
# Plot ablation importance

plt.figure(figsize=(8, 4))
plt.bar(ablation_df["ablation"], ablation_df["delta_vs_full"])
plt.axhline(0, linestyle="--")
plt.ylabel("RMSE increase vs full model")
plt.title("Metadata importance by ablation")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## Choose a meta-law winner

We compare direct and latent models by mean coefficient RMSE and pick the best overall winner.

In [ ]:
# Pick best meta-law candidate

all_mean_rmse = pd.concat([
    coef_pred_scores.groupby("model")["rmse"].mean(),
    latent_scores.groupby("model")["rmse"].mean(),
]).sort_values()

display(all_mean_rmse)

meta_model_name = all_mean_rmse.index[0]
print("Best meta-law model:", meta_model_name)

predictions_map = {
    "linear": pred_linear,
    "ridge": pred_ridge,
    "mlp_small": pred_mlp,
    "latent_linear": recon_linear,
    "latent_ridge": recon_ridge,
    "latent_mlp_small": recon_mlp,
}
Yhat_meta = predictions_map[meta_model_name]
Yhat_meta_df = pd.DataFrame(Yhat_meta, columns=COEF_COLS)

## Field reconstruction against empirical flow

The earlier notebook compared meta-law fields mostly to regime-specific template fields. This upgrade
adds direct comparisons against the **empirical observed flow** using:

- RMSE
- explained variance
- Pearson correlation

for:
- shared-law
- meta-law
- regime-specific

In [ ]:
# Field reconstruction metrics against empirical flow

def safe_corr(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    if np.std(a) < 1e-12 or np.std(b) < 1e-12:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])

def explained_variance_score_manual(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.var(y_true)
    if denom < 1e-12:
        return np.nan
    return float(1.0 - np.var(y_true - y_pred) / denom)

field_rows = []
for i, row in coef_df.iterrows():
    rid = row["regime_id"]
    sub = regime_subsets[rid]
    y_emp = sub["predicted_flow"].to_numpy(dtype=float)

    beta_specific = row[COEF_COLS].to_numpy(dtype=float)
    beta_meta = Yhat_meta_df.iloc[i].to_numpy(dtype=float)

    pred_shared = predict_with_beta(sub, beta_shared)
    pred_meta = predict_with_beta(sub, beta_meta)
    pred_specific = predict_with_beta(sub, beta_specific)

    field_rows.append({
        "regime_id": rid,
        "shared_field_rmse": math.sqrt(mean_squared_error(y_emp, pred_shared)),
        "meta_field_rmse": math.sqrt(mean_squared_error(y_emp, pred_meta)),
        "specific_field_rmse": math.sqrt(mean_squared_error(y_emp, pred_specific)),
        "shared_field_ev": explained_variance_score_manual(y_emp, pred_shared),
        "meta_field_ev": explained_variance_score_manual(y_emp, pred_meta),
        "specific_field_ev": explained_variance_score_manual(y_emp, pred_specific),
        "shared_field_corr": safe_corr(y_emp, pred_shared),
        "meta_field_corr": safe_corr(y_emp, pred_meta),
        "specific_field_corr": safe_corr(y_emp, pred_specific),
    })

field_compare_df = pd.DataFrame(field_rows)
display(field_compare_df.head())

print("Mean field RMSE:")
print(field_compare_df[["shared_field_rmse", "meta_field_rmse", "specific_field_rmse"]].mean())
print("\nMean field explained variance:")
print(field_compare_df[["shared_field_ev", "meta_field_ev", "specific_field_ev"]].mean())
print("\nMean field correlation:")
print(field_compare_df[["shared_field_corr", "meta_field_corr", "specific_field_corr"]].mean())

In [ ]:
# Plot field reconstruction summary

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(["shared", "meta", "specific"], field_compare_df[["shared_field_rmse", "meta_field_rmse", "specific_field_rmse"]].mean())
axes[0].set_title("Mean field RMSE")

axes[1].bar(["shared", "meta", "specific"], field_compare_df[["shared_field_ev", "meta_field_ev", "specific_field_ev"]].mean())
axes[1].set_title("Mean field explained variance")

axes[2].bar(["shared", "meta", "specific"], field_compare_df[["shared_field_corr", "meta_field_corr", "specific_field_corr"]].mean())
axes[2].set_title("Mean field correlation")

plt.tight_layout()
plt.show()

In [ ]:
# Scatter: shared-law vs meta-law field RMSE by regime

plt.figure(figsize=(7, 5))
plt.scatter(field_compare_df["shared_field_rmse"], field_compare_df["meta_field_rmse"])
mn = min(field_compare_df["shared_field_rmse"].min(), field_compare_df["meta_field_rmse"].min())
mx = max(field_compare_df["shared_field_rmse"].max(), field_compare_df["meta_field_rmse"].max())
plt.plot([mn, mx], [mn, mx], linestyle="--")
for _, r in field_compare_df.iterrows():
    plt.text(r["shared_field_rmse"], r["meta_field_rmse"], r["regime_id"].split("|")[2], fontsize=7, alpha=0.7)
plt.xlabel("shared-law field RMSE")
plt.ylabel("meta-law field RMSE")
plt.title("Field reconstruction: shared-law vs meta-law")
plt.tight_layout()
plt.show()

## Trajectory reconstruction from predicted coefficients

We compare three coefficient choices for each regime:
- shared-law
- meta-law
- regime-specific

In [ ]:
# Shared-law coefficient vector

Y_all = Y_coef.to_numpy(dtype=float)
beta_shared = np.mean(Y_all, axis=0)

def trajectory_gap(sub, beta_ref, beta_cmp, n_r0=15):
    cmin, cmax = sub["condition_coord"].min(), sub["condition_coord"].max()
    rmin, rmax = sub["residual"].min(), sub["residual"].max()
    cgrid = np.linspace(cmin, cmax, 40)
    r0s = np.linspace(np.quantile(sub["residual"], 0.05), np.quantile(sub["residual"], 0.95), n_r0)
    flow_cap = max(1.0, 2.5 * np.quantile(np.abs(sub["predicted_flow"]), 0.995))

    def integrate(beta, r0):
        r = float(np.clip(r0, rmin, rmax))
        traj = [r]
        for i in range(len(cgrid) - 1):
            c = cgrid[i]
            dc = float(cgrid[i+1] - cgrid[i])
            x = np.array([1.0, c, r, c**3, r**2, r * c**2], dtype=float)
            g = float(np.clip(x @ beta, -flow_cap, flow_cap))
            r = float(np.clip(r + g * dc, rmin - 0.5, rmax + 0.5))
            traj.append(r)
        return np.array(traj)

    errs = []
    for r0 in r0s:
        t_ref = integrate(beta_ref, r0)
        t_cmp = integrate(beta_cmp, r0)
        errs.append(math.sqrt(mean_squared_error(t_ref, t_cmp)))
    return float(np.mean(errs))

traj_rows = []
for i, row in coef_df.iterrows():
    beta_true = row[COEF_COLS].to_numpy(dtype=float)
    beta_meta = Yhat_meta_df.iloc[i].to_numpy(dtype=float)
    sub = regime_subsets[row["regime_id"]]

    traj_rows.append({
        "regime_id": row["regime_id"],
        "system": row["system"],
        "task": row["task"],
        "forcing_mode": row["forcing_mode"],
        "k": row["k"],
        "flow_mode": row["flow_mode"],
        "shared_vs_specific": trajectory_gap(sub, beta_true, beta_shared),
        "meta_vs_specific": trajectory_gap(sub, beta_true, beta_meta),
    })

traj_compare_df = pd.DataFrame(traj_rows)
display(traj_compare_df.head())

In [ ]:
# Cleaner trajectory summary by forcing mode

traj_force = traj_compare_df.groupby("forcing_mode")[["shared_vs_specific", "meta_vs_specific"]].mean().reset_index()

x = np.arange(len(traj_force))
w = 0.38
plt.figure(figsize=(8, 4))
plt.bar(x - w/2, traj_force["shared_vs_specific"], width=w, label="shared-law vs regime-specific")
plt.bar(x + w/2, traj_force["meta_vs_specific"], width=w, label="meta-law vs regime-specific")
plt.xticks(x, traj_force["forcing_mode"])
plt.ylabel("mean trajectory RMSE vs regime-specific")
plt.title("Trajectory reconstruction by forcing mode")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cleaner trajectory summary by system and flow mode

traj_sys = traj_compare_df.groupby(["system", "flow_mode"])[["shared_vs_specific", "meta_vs_specific"]].mean().reset_index()
traj_sys["label"] = traj_sys["system"].astype(str) + "|" + traj_sys["flow_mode"].astype(str)

x = np.arange(len(traj_sys))
w = 0.38
plt.figure(figsize=(10, 4))
plt.bar(x - w/2, traj_sys["shared_vs_specific"], width=w, label="shared-law")
plt.bar(x + w/2, traj_sys["meta_vs_specific"], width=w, label="meta-law")
plt.xticks(x, traj_sys["label"], rotation=30, ha="right")
plt.ylabel("mean trajectory RMSE vs regime-specific")
plt.title("Trajectory reconstruction by system | flow mode")
plt.legend()
plt.tight_layout()
plt.show()

## Selected regime visual comparison

For one representative regime, compare trajectory families under:
- shared-law
- meta-law
- regime-specific

In [ ]:
# Representative regime comparison

rep_idx = int(np.argmin(traj_compare_df["meta_vs_specific"].to_numpy()))
rep_regime_id = traj_compare_df.iloc[rep_idx]["regime_id"]
rep_row = coef_df[coef_df["regime_id"] == rep_regime_id].iloc[0]
rep_sub = regime_subsets[rep_regime_id]

beta_true = rep_row[COEF_COLS].to_numpy(dtype=float)
rep_pos = coef_df.index[coef_df["regime_id"] == rep_regime_id][0]
beta_meta = Yhat_meta_df.iloc[rep_pos].to_numpy(dtype=float)

cmin, cmax = rep_sub["condition_coord"].min(), rep_sub["condition_coord"].max()
rmin, rmax = rep_sub["residual"].min(), rep_sub["residual"].max()
cgrid = np.linspace(cmin, cmax, 45)
r0s = np.linspace(np.quantile(rep_sub["residual"], 0.1), np.quantile(rep_sub["residual"], 0.9), 8)
flow_cap = max(1.0, 2.5 * np.quantile(np.abs(rep_sub["predicted_flow"]), 0.995))

def integrate_beta(beta, r0):
    r = float(np.clip(r0, rmin, rmax))
    traj = [r]
    for i in range(len(cgrid) - 1):
        c = cgrid[i]
        dc = float(cgrid[i+1] - cgrid[i])
        x = np.array([1.0, c, r, c**3, r**2, r * c**2], dtype=float)
        g = float(np.clip(x @ beta, -flow_cap, flow_cap))
        r = float(np.clip(r + g * dc, rmin - 0.5, rmax + 0.5))
        traj.append(r)
    return np.array(traj)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for r0 in r0s:
    axes[0].plot(cgrid, integrate_beta(beta_shared, r0))
    axes[1].plot(cgrid, integrate_beta(beta_meta, r0))
    axes[2].plot(cgrid, integrate_beta(beta_true, r0))

axes[0].set_title("Shared-law")
axes[1].set_title(f"Meta-law ({meta_model_name})")
axes[2].set_title("Regime-specific")

for ax in axes:
    ax.set_xlabel("condition coordinate c")
axes[0].set_ylabel("residual")
fig.suptitle(f"Representative regime: {rep_regime_id}", y=1.03)
plt.tight_layout()
plt.show()

## Three-way comparison summary

We summarize:
- direct coefficient prediction error
- field reconstruction error
- trajectory reconstruction error

for:
- shared-law
- meta-law
- regime-specific baseline

In [ ]:
# Three-way summary table

coef_rmse_shared = math.sqrt(mean_squared_error(Y_coef.to_numpy().ravel(), np.tile(beta_shared, (len(Y_coef), 1)).ravel()))
coef_rmse_meta = math.sqrt(mean_squared_error(Y_coef.to_numpy().ravel(), Yhat_meta_df.to_numpy().ravel()))

summary_threeway = pd.DataFrame([
    {
        "method": "shared_law",
        "coefficient_rmse": coef_rmse_shared,
        "field_rmse_mean": field_compare_df["shared_field_rmse"].mean(),
        "field_ev_mean": field_compare_df["shared_field_ev"].mean(),
        "field_corr_mean": field_compare_df["shared_field_corr"].mean(),
        "trajectory_rmse_mean": traj_compare_df["shared_vs_specific"].mean(),
    },
    {
        "method": f"meta_law::{meta_model_name}",
        "coefficient_rmse": coef_rmse_meta,
        "field_rmse_mean": field_compare_df["meta_field_rmse"].mean(),
        "field_ev_mean": field_compare_df["meta_field_ev"].mean(),
        "field_corr_mean": field_compare_df["meta_field_corr"].mean(),
        "trajectory_rmse_mean": traj_compare_df["meta_vs_specific"].mean(),
    },
    {
        "method": "regime_specific",
        "coefficient_rmse": 0.0,
        "field_rmse_mean": field_compare_df["specific_field_rmse"].mean(),
        "field_ev_mean": field_compare_df["specific_field_ev"].mean(),
        "field_corr_mean": field_compare_df["specific_field_corr"].mean(),
        "trajectory_rmse_mean": 0.0,
    },
])

display(summary_threeway)

In [ ]:
# Three-way comparison plot

fig, axes = plt.subplots(1, 5, figsize=(18, 4))

metrics = [
    "coefficient_rmse",
    "field_rmse_mean",
    "field_ev_mean",
    "field_corr_mean",
    "trajectory_rmse_mean",
]
titles = [
    "Coefficient RMSE",
    "Field RMSE",
    "Field explained variance",
    "Field correlation",
    "Trajectory RMSE",
]

for ax, m, t in zip(axes, metrics, titles):
    ax.bar(summary_threeway["method"], summary_threeway[m])
    ax.set_title(t)
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## Final verdict table

Combine regime diagnostics with meta-law reconstruction quality.

In [ ]:
# Final verdicts

final_df = coef_df.merge(field_compare_df, on="regime_id", how="left").merge(traj_compare_df, on="regime_id", how="left")

def verdict(row):
    if row["meta_field_corr"] > row["shared_field_corr"] and row["meta_vs_specific"] < row["shared_vs_specific"]:
        if row["meta_field_corr"] > 0.95:
            return "meta-law successful"
        return "latent/direct meta-law good"
    if row["meta_vs_specific"] < row["shared_vs_specific"]:
        return "shared-law too coarse"
    return "regime-specific still needed"

final_df["verdict"] = final_df.apply(verdict, axis=1)

display(final_df[[
    "regime_id", "r2", "rmse", "additive_r2", "fb_gap",
    "shared_field_rmse", "meta_field_rmse", "specific_field_rmse",
    "shared_field_corr", "meta_field_corr", "specific_field_corr",
    "shared_vs_specific", "meta_vs_specific", "verdict"
] + COEF_COLS].head(20))

## Working conclusion

Notebook 48 now evaluates meta-law quality against the empirical flow directly, not only against the
regime-specific template field.

A strong result is:

- coefficient vectors are predictable from metadata
- latent-space prediction competes with direct prediction
- meta-law fields are much closer to empirical flow than a shared-law
- meta-law trajectories improve substantially over a single shared-law

If that pattern holds on your real exports, the next notebook should be:

**Notebook 49 — meta-law generalization with holdout-regime tests**

where one or more full regimes are held out entirely during training.